# Quantum Radiance Fields (QRF) Tutorial

This tutorial provides a comprehensive overview of the Quantum Radiance Fields (QRF) framework, covering encoding strategies, parameterized quantum circuits, full model pipeline, 2D image regression, and 3D volume rendering with density visualization.

## 1. Installation
Install required dependencies:

In [ ]:
!pip install pennylane pennylane-borealis matplotlib torch omegaconf hydra-core

## 2. Imports
Import core QRF modules and libraries:

In [ ]:
import numpy as np
import pennylane as qml
import matplotlib.pyplot as plt
from qrf import (
    angle_encoding, dense_angle_encoding,
    wavefunction_encoding, general_qubit_encoding,
    apply_circuit_a, apply_circuit_b,
    apply_circuit_c, apply_circuit_d,
    qrf_model, sample_ray, render_one_ray,
    accumulate_volume_rgb, QRFForward
)

## 3. Encoding Demonstration
Four encoding strategies:

- **angle_encoding**: one feature per qubit using `RY`
- **dense_angle_encoding**: two features per qubit using `RY + RZ`
- **wavefunction_encoding**: full state preparation via `MottonenStatePreparation`
- **general_qubit_encoding**: three features per qubit mapped by `RX, RY, RZ`

In [ ]:
dev = qml.device('default.qubit', wires=3)
features = np.array([0.1, 0.2, 0.3, 0.4, 0.5, 0.6])
@qml.qnode(dev)
def demo_encoding(encode_fn):
    encode_fn(features, wires=range(3))
    return [qml.expval(qml.PauliZ(w)) for w in range(3)]

print('Angle encoding output:', demo_encoding(angle_encoding))
print('Dense angle encoding output:', demo_encoding(dense_angle_encoding))
print('Wavefunction encoding output:', demo_encoding(wavefunction_encoding))
print('General qubit encoding output:', demo_encoding(general_qubit_encoding))

## 4. Quantum Circuit Examples
QRF supports four parameterized circuits: A, B, C, D.

In [ ]:
dev2 = qml.device('default.qubit', wires=6)
params = np.random.rand(20)
@qml.qnode(dev2)
def demo_circuit(circuit_fn):
    dense_angle_encoding(np.random.rand(6), wires=range(6))
    circuit_fn(params, wires=range(6))
    # Measure qubit 0/1/2 for R,G,B and qubit 3 for sigma
    return [qml.expval(qml.PauliZ(i)) for i in [0,1,2,3]]

for fn in [apply_circuit_a, apply_circuit_b, apply_circuit_c, apply_circuit_d]:
    print(f"{fn.__name__} output:", demo_circuit(fn))

## 5. QRFForward Pipeline
Encapsulate encoding, circuit, activation, and measurement.

In [ ]:
# Initialize QRFForward model
model = QRFForward(circuit='circuit_a', encoding='dense_angle')
params = np.random.rand(20)  # example parameter vector length
model.set_params(params)
# Inference at a single point
xyz = np.array([0.3, 0.5, 0.7])
view = np.array([0.6, 0.4])  # theta, phi
r, g, b, sigma = model.infer_point(xyz, view, delta=0.01)
print('Point output -> R,G,B,sigma:', r, g, b, sigma)

## 6. 2D Image Regression Example
Use QRF to regress 2D coordinates to RGB values.

In [ ]:
grid = np.linspace(0, 1, 30)
image = np.zeros((30, 30, 3))
for i, x in enumerate(grid):
    for j, y in enumerate(grid):
        r, g, b, _ = model.infer_point(np.array([x, y, 0.0]), np.array([0.5, 0.5]), 0.01)
        image[j, i] = [r, g, b]
plt.figure(figsize=(5,5))
plt.imshow(image)
plt.title('2D Regression Prediction')
plt.axis('off')

## 7. 3D Volume Rendering Example
Render a single ray with density visualization.

In [ ]:
origin = np.array([0.5, 0.5, -0.5])
direction = np.array([0, 0, 1])
rgb = model.render_ray(origin, direction, n_samples=40, visualize_sigma=True)
print('Rendered RGB:', rgb)

## Conclusion
This tutorial demonstrated the full QRF workflow. For training and advanced experiments, refer to `train.py` and configuration files.